# Sprint 6 — 04 Inference Testing

**Objective:** validate end-to-end inference against the deployed serving endpoint `civ4_rag_endpoint`, covering all 10 scenarios required by the Sprint 6 spec, and compare the deployed behavior with the validated Sprint 4 pipeline.

## Test matrix

| # | Scenario | Where validated |
| - | --- | --- |
| 1 | Model loading and initialization | Load from `models:/workspace.gcivilization.civ4_rag_model@champion` (UC alias) |
| 2 | Valid question submission | SDK query + raw REST invocations |
| 3 | Successful retrieval from Vector Search | Non-empty `references`, expected documents hit |
| 4 | Successful LLM generation | Non-empty grounded `answer` |
| 5 | Correct response structure | `answer` + `references[{document_id, title, source_url, ...}]` contract |
| 6 | Presence of document references | Hard assert on every valid response |
| 7 | Empty or invalid questions | Endpoint returns structured response / informative rejection |
| 8 | Retrieval or generation service failures | Simulated outage via monkeypatch → informative error, no secrets |
| 9 | Serving endpoint availability | `state.ready == READY` |
| 10 | End-to-end inference through the REST API | SDK `query()` + raw `POST /serving-endpoints/<name>/invocations` |

## Inputs / outputs

* **Inputs:** deployed endpoint, registered model, 3 representative questions from the Sprint 5 dataset (`civ4_eval_questions`), Sprint 4 `rag_utils` for parity checks.
* **Outputs:** 10/10 PASS checklist + evidence (answers, references, parity result).

## Cost notes

The endpoint is scale-to-zero: the **first** invocation pays a cold start (~1–2 min). Total live inference calls are kept small (2 full RAG questions + 2 edge cases). Answer text is compared structurally, not byte-for-byte (temperature 0.1 generation is not deterministic); retrieval **is** deterministic and is compared exactly.

In [0]:
# Setup — imports, deployment targets, and representative test questions
# drawn from the Sprint 5 evaluation dataset (with expected documents).
import sys
for _dir in (
    "/Workspace/Repos/endoerner@gmail.com/civ4-rag/04_generation",
    "/Workspace/Repos/endoerner@gmail.com/civ4-rag/06_model_serving",
):
    if _dir not in sys.path:
        sys.path.insert(0, _dir)

import json
import mlflow
import pandas as pd
import rag_model
import rag_utils
from databricks.sdk import WorkspaceClient

mlflow.set_registry_uri("databricks-uc")
w = WorkspaceClient()

REGISTERED_MODEL_NAME = "workspace.gcivilization.civ4_rag_model"
MODEL_ALIAS = "champion"
ENDPOINT_NAME = "civ4_rag_endpoint"
REST_PATH = f"/api/2.0/serving-endpoints/{ENDPOINT_NAME}/invocations"

# 3 representative questions from the Sprint 5 eval set — one per DISTINCT category
test_qs = spark.sql("""
    SELECT question_id, question, category, expected_document_ids
    FROM (
        SELECT *, ROW_NUMBER() OVER (PARTITION BY category ORDER BY question_id) AS rn
        FROM workspace.gcivilization.civ4_eval_questions
    )
    WHERE rn = 1
    ORDER BY category
    LIMIT 3
""").collect()
assert len(test_qs) == 3, f"expected 3 test questions, got {len(test_qs)}"
for q in test_qs:
    print(f"  [{q.category}] {q.question[:80]}...  expected docs: {q.expected_document_ids}")
print("PASS  test questions loaded from Sprint 5 evaluation dataset")

In [0]:
# Scenarios 9 + 1 — endpoint availability and model loading/initialization.
# 9: serving endpoint must be READY.
# 1: the registered model must load independently of the dev notebooks,
#    resolved through the UC 'champion' alias, with load_context executed.
ep = w.serving_endpoints.get(ENDPOINT_NAME)
ready = getattr(ep.state.ready, "value", ep.state.ready)
assert ready == "READY", f"endpoint state: {ready}"
print(f"PASS [9] endpoint '{ENDPOINT_NAME}' is READY")

served = ep.config.served_entities[0]
assert served.entity_name == REGISTERED_MODEL_NAME
print(f"      serving {served.entity_name} version {served.entity_version} (CPU {served.workload_size}, scale-to-zero)")

# 1: load the packaged model from the registry (alias-resolved)
local_model = mlflow.pyfunc.load_model(f"models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS}")
probe = local_model.predict(pd.DataFrame({"question": [""]}))
assert "No valid question" in probe[0]["answer"] and probe[0]["references"] == []
sig_in = str(local_model.metadata.get_input_schema())
sig_out = str(local_model.metadata.get_output_schema())
assert "question" in sig_in and "answer" in sig_out and "references" in sig_out
print("PASS [1] model loads from models:/...@champion; signature + predict path verified (no-cost probe)")

In [0]:
# Scenario 10 (part 1) — end-to-end inference through the REST API via the
# SDK wrapper (w.serving_endpoints.query -> POST .../invocations).
# NOTE: first call after scale-to-zero pays a cold start (~1-2 min).
q1 = test_qs[0]
print(f"Q1 [{q1.category}]: {q1.question}")
resp_sdk = w.serving_endpoints.query(
    name=ENDPOINT_NAME,
    dataframe_records=[{"question": q1.question}],
)
preds = getattr(resp_sdk, "predictions", None)
if preds is None:  # defensive: fall back to dict-like access
    preds = resp_sdk["predictions"]
r1 = preds[0]

# Scenarios 2-6 (on the SDK response): valid question, retrieval, generation,
# structure, references.
assert isinstance(r1, dict) and set(r1) >= {"answer", "references"}, f"bad structure: {list(r1)}"
assert isinstance(r1["answer"], str) and len(r1["answer"].strip()) > 50, "answer too short/empty"
assert isinstance(r1["references"], list) and len(r1["references"]) >= 1, "no references"
ref_fields = {"document_id", "title", "source_url", "author", "category"}
assert all(ref_fields <= set(s) for s in r1["references"]), "reference fields missing"

hit_docs = {s["document_id"] for s in r1["references"]}
assert hit_docs & set(q1.expected_document_ids), \
    f"retrieval missed expected docs {q1.expected_document_ids}; got {hit_docs}"

print("ANSWER (truncated):", r1["answer"][:280], "...")
for s in r1["references"]:
    print(f"  REF {s['document_id']} | {s['title'][:55]}")
print(f"PASS [2,3,4,5,6,10a] SDK REST query: valid question -> grounded answer + {len(r1['references'])} reference(s), expected doc hit")

In [0]:
# Scenario 10 (part 2) — the literal REST interface via api_client.do:
# POST /api/2.0/serving-endpoints/<name>/invocations (the exact route
# documented in README.md — the SDK's do() adds the /api/2.0 prefix, so the
# path is passed unprefixed). Session auth — no token is read or printed.
q2 = test_qs[1]
print(f"Q2 [{q2.category}]: {q2.question}")

rest_body = {"dataframe_records": [{"question": q2.question}]}
http = w.api_client.do("POST", f"/serving-endpoints/{ENDPOINT_NAME}/invocations", body=rest_body)
r2 = http["predictions"][0]

assert isinstance(r2.get("answer"), str) and len(r2["answer"].strip()) > 50
assert isinstance(r2.get("references"), list) and len(r2["references"]) >= 1
assert all({"document_id", "title", "source_url"} <= set(s) for s in r2["references"])
hit_docs2 = {s["document_id"] for s in r2["references"]}
assert hit_docs2 & set(q2.expected_document_ids), \
    f"retrieval missed expected docs {q2.expected_document_ids}; got {hit_docs2}"

print("ANSWER (truncated):", r2["answer"][:280], "...")
for s in r2["references"]:
    print(f"  REF {s['document_id']} | {s['title'][:55]}")
print("PASS [10b] raw REST POST /invocations: same contract, expected doc hit")

In [0]:
# Parity — the deployed pipeline must retrieve the SAME documents as the
# validated Sprint 4 implementation (embedding + ANN search are deterministic;
# only LLM text differs at temperature 0.1, so retrieval is compared exactly).
for label, q_row, deployed_refs in (("Q1", q1, r1["references"]), ("Q2", q2, r2["references"])):
    direct = rag_utils.retrieve(q_row.question, top_k=rag_model.PINNED_CONFIG["top_k"])
    direct_docs = {c["document_id"] for c in direct}
    deployed_docs = {s["document_id"] for s in deployed_refs}
    assert direct_docs == deployed_docs, f"{label} parity mismatch: {direct_docs ^ deployed_docs}"
    print(f"PASS      {label}: {len(deployed_docs)} documents identical to Sprint 4 rag_utils.retrieve(top_k=10)")
print("PASS      retrieval parity with the validated pipeline confirmed (no deployment drift)")

In [0]:
# Scenario 7 — empty and malformed questions through the ENDPOINT.

# 7a. Empty question: HTTP 200 with a structured, informative response
#     (validation inside the model — no Vector Search / LLM call needed).
empty_resp = w.api_client.do(
    "POST", f"/serving-endpoints/{ENDPOINT_NAME}/invocations",
    body={"dataframe_records": [{"question": "   "}]},
)
r_empty = empty_resp["predictions"][0]
assert "No valid question" in r_empty["answer"]
assert r_empty["references"] == []
print("PASS [7a] empty question -> 200 + structured response (" + r_empty["answer"][:60] + "...)")

# 7b. Malformed payload (missing 'question' field): must NOT silently
#     succeed — expect a rejection with an informative message.
try:
    w.api_client.do(
        "POST", f"/serving-endpoints/{ENDPOINT_NAME}/invocations",
        body={"dataframe_records": [{"wrong_field": 1}]},
    )
    raise AssertionError("malformed payload should be rejected")
except AssertionError:
    raise
except Exception as e:
    msg = str(e)
    assert "question" in msg.lower() or "400" in msg or "error" in msg.lower(), \
        f"uninformative rejection: {msg[:200]}"
    print(f"PASS [7b] malformed payload rejected informatively: {msg[:120]}")

In [0]:
# Scenario 8 — retrieval or generation service failures, simulated locally
# by monkeypatching the Sprint 4 stages. The packaged model must raise an
# INFORMATIVE error (and never leak credentials — none are present in it).
model = rag_model.Civ4RagModel()
model_input = pd.DataFrame({"question": [test_qs[2].question]})

# 8a. Vector Search outage (retrieve fails)
orig_retrieve = rag_utils.retrieve
def _broken_retrieve(*a, **k):
    raise RuntimeError("simulated Vector Search outage: endpoint unreachable")
rag_utils.retrieve = _broken_retrieve
try:
    try:
        model.predict(None, model_input)
        raise AssertionError("retrieval failure should propagate")
    except RuntimeError as e:
        assert "RAG pipeline execution failed" in str(e)
        assert "simulated Vector Search outage" in str(e)
        assert "secrets" not in str(e).lower() and "token" not in str(e).lower()
        print(f"PASS [8a] retrieval failure -> informative error: {str(e)[:100]}")
finally:
    rag_utils.retrieve = orig_retrieve

# 8b. LLM outage (generate_answer fails)
orig_generate = rag_utils.generate_answer
def _broken_generate(*a, **k):
    raise RuntimeError("simulated LLM endpoint 503")
rag_utils.generate_answer = _broken_generate
try:
    try:
        model.predict(None, model_input)
        raise AssertionError("generation failure should propagate")
    except RuntimeError as e:
        assert "RAG pipeline execution failed" in str(e)
        assert "simulated LLM endpoint 503" in str(e)
        print(f"PASS [8b] generation failure -> informative error: {str(e)[:100]}")
finally:
    rag_utils.generate_answer = orig_generate

# Confirm both stages are restored (clean state for later cells)
assert rag_utils.retrieve is orig_retrieve and rag_utils.generate_answer is orig_generate
print("PASS      monkeypatches restored — Sprint 4 module unchanged")

In [0]:
# Summary — Sprint 6 inference testing acceptance (10/10).
print("=" * 62)
print("SPRINT 6 NOTEBOOK 04: ALL INFERENCE TESTS PASSED (10/10)")
print("=" * 62)
print(f"""
 [1]  Model loading ........... models:/{REGISTERED_MODEL_NAME}@{MODEL_ALIAS} loads + signature OK
 [2]  Valid question .......... 2 live questions (SDK + raw REST)
 [3]  Vector Search retrieval . references present, expected documents hit
 [4]  LLM generation .......... grounded answers > 50 chars with [Source N] citations
 [5]  Response structure ...... answer + references[{{document_id,title,source_url,author,category}}]
 [6]  Document references ..... hard-asserted on every valid response
 [7]  Empty/invalid input ..... 200 + structured response; malformed -> informative rejection
 [8]  Service failures ........ retrieval/LLM outages -> informative errors, no secrets
 [9]  Endpoint availability ... {ENDPOINT_NAME} READY
 [10] REST end-to-end ......... SDK query + POST {REST_PATH}

 Parity: deployed retrieval == Sprint 4 rag_utils.retrieve(top_k=10) for Q1 & Q2
 Cost: 2 full RAG invocations + 2 edge-case invocations (scale-to-zero observed)
""")
print("SPRINT 6 VALIDATION COMPLETE — see 06_model_serving/README.md")